In [1]:
import sys
import pandas as pd
import numpy as np

print("Python:", sys.executable)
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)

Python: /home/sirzazo/master/ProyectoMaster/.venv-busmatch/bin/python
Pandas: 3.1.1
NumPy: 2.5.5


In [2]:
from pathlib import Path
import pandas as pd

# Ruta del CSV comprimido
archivo = Path(
    "/home/sirzazo/master/ProyectoMaster/TFM-Evolve/"
    "docs/exploracion_datos/BusMatch/datos_busmatch/raw/"
    "20241030_Viajes_distritos.csv.gz"
)

# Ruta donde guardaremos los datos filtrados
salida = archivo.parent.parent / "silver" / "viajes_toledo.csv"
salida.parent.mkdir(parents=True, exist_ok=True)

print("¿Existe el archivo?", archivo.exists())

if archivo.exists():

    print(f"Tamaño: {archivo.stat().st_size / 1024**2:.2f} MB")

    total_toledo = 0
    primera_escritura = True

    # Leer el archivo completo por bloques
    for bloque in pd.read_csv(
        archivo,
        sep="|",
        compression="gzip",
        dtype={"origen": "string", "destino": "string"},
        chunksize=100_000
    ):

        # Filtrar viajes con origen O destino en Toledo
        filtro = (
            bloque["origen"].str.startswith("45", na=False) |
            bloque["destino"].str.startswith("45", na=False)
        )

        df_filtrado = bloque.loc[filtro]

        # Guardar los registros directamente en el CSV
        if not df_filtrado.empty:

            df_filtrado.to_csv(
                salida,
                sep="|",
                mode="w" if primera_escritura else "a",
                header=primera_escritura,
                index=False
            )

            primera_escritura = False
            total_toledo += len(df_filtrado)

    print("\nRegistros relacionados con Toledo:", total_toledo)

    if total_toledo > 0:
        print("Archivo guardado en:", salida)

        # Mostrar únicamente 10 filas para comprobar el resultado
        muestra = pd.read_csv(
            salida,
            sep="|",
            nrows=10,
            dtype={"origen": "string", "destino": "string"}
        )

        display(muestra)
        print("Columnas:", muestra.columns.tolist())
        print("Dimensiones de la muestra:", muestra.shape)

¿Existe el archivo? True
Tamaño: 205.48 MB

Registros relacionados con Toledo: 373361
Archivo guardado en: /home/sirzazo/master/ProyectoMaster/TFM-Evolve/docs/exploracion_datos/BusMatch/datos_busmatch/silver/viajes_toledo.csv


,fecha,periodo,origen,destino,distancia,actividad_origen,actividad_destino,estudio_origen_posible,estudio_destino_posible,residencia,renta,edad,sexo,viajes,viajes_km
0,20241030,18,45101_AM,0200302,>50,frecuente,casa,no,no,2,10-15,25-45,mujer,3.763,424.991
1,20241030,16,4516805,0200303,>50,frecuente,casa,no,no,2,10-15,0-25,hombre,5.198,1099.456
2,20241030,20,4508101,0200304,>50,frecuente,casa,no,no,2,10-15,65-100,NaN,2.437,515.583
3,20241030,20,45185,0200304,>50,frecuente,casa,no,no,2,10-15,65-100,NaN,3.695,536.658
4,20241030,12,4516807,0200306,>50,frecuente,casa,no,no,2,>15,0-25,NaN,7.126,1462.505
5,20241030,11,45135,0200306,>50,frecuente,casa,no,no,2,>15,45-65,mujer,4.354,558.769
6,20241030,10,45084,0200307,>50,frecuente,casa,no,no,2,10-15,45-65,mujer,3.492,526.510
7,20241030,4,45123,02009,>50,frecuente,casa,no,no,2,10-15,0-25,hombre,4.070,1007.424
8,20241030,19,45186,02012_AM,>50,frecuente,casa,no,no,2,<10,NaN,NaN,2.167,233.984
9,20241030,17,45186,02019_AM,>50,frecuente,casa,no,no,2,10-15,NaN,NaN,4.066,376.124


Columnas: ['fecha', 'periodo', 'origen', 'destino', 'distancia', 'actividad_origen', 'actividad_destino', 'estudio_origen_posible', 'estudio_destino_posible', 'residencia', 'renta', 'edad', 'sexo', 'viajes', 'viajes_km']
Dimensiones de la muestra: (10, 15)


In [11]:
import pandas as pd

df_toledo = pd.read_csv(
    "datos_busmatch/silver/viajes_toledo.csv",
    sep="|",
    dtype={"origen": str, "destino": str}
)
print("Dimensiones del DataFrame de Toledo:", df_toledo.shape)

toledo_trabajo = df_toledo[df_toledo["actividad_destino"] == "trabajo_estudio"]
print(toledo_trabajo.info())
display(toledo_trabajo.head(100))
print(toledo_trabajo.shape[0])

Dimensiones del DataFrame de Toledo: (373361, 15)
<class 'pandas.DataFrame'>
RangeIndex: 47054 entries, 326307 to 373360
Data columns (total 15 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   fecha                    47054 non-null  int64  
 1   periodo                  47054 non-null  int64  
 2   origen                   47054 non-null  str    
 3   destino                  47054 non-null  str    
 4   distancia                47054 non-null  str    
 5   actividad_origen         47054 non-null  str    
 6   actividad_destino        47054 non-null  str    
 7   estudio_origen_posible   47054 non-null  str    
 8   estudio_destino_posible  47054 non-null  str    
 9   residencia               47054 non-null  int64  
 10  renta                    47054 non-null  str    
 11  edad                     21001 non-null  str    
 12  sexo                     14811 non-null  str    
 13  viajes                   47054 

,fecha,periodo,origen,destino,distancia,actividad_origen,actividad_destino,estudio_origen_posible,estudio_destino_posible,residencia,renta,edad,sexo,viajes,viajes_km
326307,20241030,5,45054,0208101,>50,casa,trabajo_estudio,no,no,45,<10,NaN,NaN,4.200,318.728
326308,20241030,7,4508701,0208103,>50,casa,trabajo_estudio,no,no,45,10-15,NaN,NaN,2.696,224.243
326309,20241030,5,45013_AM,05002_AM,10-50,casa,trabajo_estudio,no,no,45,10-15,NaN,NaN,1.273,32.464
326310,20241030,0,45045_AM,05002_AM,10-50,casa,trabajo_estudio,no,no,45,<10,NaN,NaN,1.960,35.336
326311,20241030,3,45045_AM,05002_AM,10-50,casa,trabajo_estudio,no,no,45,10-15,NaN,NaN,2.265,42.192
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
326402,20241030,7,45082_AM,1013101,10-50,casa,trabajo_estudio,no,no,45,10-15,NaN,NaN,3.339,110.307
326403,20241030,8,45082_AM,1013101,10-50,casa,trabajo_estudio,no,no,45,10-15,NaN,NaN,3.229,94.892
326404,20241030,7,45125,1013101,10-50,casa,trabajo_estudio,no,no,45,10-15,NaN,NaN,5.758,189.043
326405,20241030,5,45040_AM,1013101,>50,casa,trabajo_estudio,no,no,45,<10,NaN,NaN,1.521,123.689


47054
